<a href="https://colab.research.google.com/github/raimund-buehler/visualizing-the-brain/blob/main/notebooks/day_one.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Day One in Colab"/></a>

# Day One: From Brain Images to a First fMRI Contrast

Today we will use Python to look at brain images, understand how they are made from numbers, and finally answer a simple research question.

By the end, you will be able to:

- describe voxels, slices, volumes, and 4D fMRI data,
- inspect an MRI image as an array of numbers,
- move through the brain with different visualizations,
- explain why an fMRI image always needs to be compared with something,
- describe the basic idea of a contrast, and
- calculate a first-level contrast for one participant.

## What is a Jupyter notebook?

A Jupyter notebook is like a website where text, code, pictures, and results can be shown together.

This notebook has two kinds of boxes. They are called **cells**:

- **Text cells**, like this one, explain what is happening.
- **Code cells** contain Python instructions that the computer can run.

To run a code cell, click the play button on the left. You can also click inside the cell and press **Shift + Enter**.

Run the cells from top to bottom. A later cell sometimes needs something that was created in an earlier cell, so it is important to run the cells in order.

## Setup: press play here first

The next cell installs and imports the tools we need for this notebook.

You do not need to understand every setup line. Run it once and wait for the message "**Setup complete**" below the cell.

In [ ]:
# @title Setup: install tools and load an anatomical template
import sys
import subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "nilearn", "nibabel", "matplotlib", "numpy", "pandas"
    ])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nilearn import datasets, image, plotting
from nilearn.glm.first_level import (
    FirstLevelModel,
    make_first_level_design_matrix,
    spm_hrf,
)

%matplotlib inline

# This is an average anatomical brain in a shared coordinate system.
anat_img = datasets.load_mni152_template(resolution=2)

print("Setup complete.")
print("Anatomical image shape:", anat_img.shape)

## MRI data: anatomy and function

MRI scanners can create different kinds of data. Today we will use two of them:

- An **anatomical MRI** is a detailed 3D image of brain structure. Each voxel has one brightness value.
- **fMRI** records many 3D brain volumes one after another. Each voxel therefore has a series of values over time.

A scanner records the brain in **slices**. A stack of slices makes one 3D **volume**. In fMRI, many volumes together make a 4D data set.

<img src="https://raw.githubusercontent.com/raimund-buehler/visualizing-the-brain/main/src/images/mri_fmri_build_up.png" alt="MRI and fMRI data built from voxels, slices, volumes, and time" width="850">

The highlighted green cube is one **voxel**. A voxel is like a 3D pixel.

## A brain image is a grid of numbers

A computer cannot directly draw a picture of a brain. First, it stores a grid of numbers. This kind of number grid is called an **array**. Plotting programs turn these numbers into brightness or color.

For a 3D anatomical image, the array has three dimensions:

- the first dimension moves through the **x** direction,
- the second through the **y** direction, and
- the third through the **z** direction.

### Read one voxel

The code below automatically finds the middle array position, so the voxel in the middle of the image. The three numbers are an **array index**. They identify a stored voxel.

We can use three array indices to ask for the value of one voxel.

In [ ]:
# Save the numbers from the image in the variable anat_data.
anat_data = anat_img.get_fdata()

# Find the middle index along x, y, and z.
middle_x = anat_data.shape[0] // 2
middle_y = anat_data.shape[1] // 2
middle_z = anat_data.shape[2] // 2

middle_value = anat_data[middle_x, middle_y, middle_z]

print("Middle array index:", (middle_x, middle_y, middle_z))
print("Value stored there:", middle_value)

An array can also give us a small block of nearby voxels. The colon in `middle_x - 1:middle_x + 2,` means: start at `middle_x - 1` and stop just before `middle_x + 2`.

So first we select a `3 × 3 × 3` block around the middle voxel. Then we draw it as a small 3D object.

In [ ]:
# Select a 3 × 3 × 3 block around the middle.
small_block = anat_data[
    middle_x - 1:middle_x + 2,
    middle_y - 1:middle_y + 2,
    middle_z - 1:middle_z + 2,
]

print("Block shape:", small_block.shape)
print(small_block)

### From numbers to a 3D object

The next cell draws the selected `3 × 3 × 3` block as cubes.

In [ ]:
# Scale the values between 0 and 1 so they can become colors.
block_range = np.ptp(small_block)
scaled_values = (small_block - small_block.min()) / block_range
voxel_colors = plt.cm.viridis(scaled_values)

# Draw all selected array positions as filled cubes.
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection="3d")
ax.voxels(
    np.ones(small_block.shape, dtype=bool),
    facecolors=voxel_colors,
    edgecolor="black",
)
ax.set_xlabel("x position")
ax.set_ylabel("y position")
ax.set_zlabel("z position")
ax.set_title(f"A {small_block.shape} array drawn as voxels")
ax.set_box_aspect((1, 1, 1))
plt.show()

### Stop and Think

Discuss these questions before moving on:

1. Do you have an idea how to change the code to create a bigger cube, for example a `4 × 4 × 4` block?
2. What do the colors in the plot mean?

<details>
<summary>Possible answers</summary>

1. One way is to change the stop value to `middle_x + 3`, `middle_y + 3`, and `middle_z + 3`. Then four positions are included: `-1`, `0`, `+1`, and `+2` around the middle.
2. The color of each voxel comes from the number stored at that position: dark purple means a lower value, bright yellow means a higher value. In an anatomical image, the value gives information about the tissue at that position (white matter, gray matter, blood vessels, etc.).

</details>

## Brain slices and directions

Scientists look at the brain in slices that are cut from three directions:

- **Coronal** or **frontal** slices separate front and back. Nilearn calls this the `y` direction.
- **Sagittal** slices separate left and right. Nilearn calls this the `x` direction.
- **Horizontal** slices separate top and bottom. Nilearn calls this the `z` direction.

<img src="https://raw.githubusercontent.com/raimund-buehler/visualizing-the-brain/main/src/images/AnatomicalPlanes.png" alt="Sagittal, coronal, and axial brain planes" width="700">

## Explore the brain

Next, we create an interactive image of the brain. Click in the different views and watch how the crosshair and the `x`, `y`, and `z` coordinates change.

In [ ]:
anat_viewer = plotting.view_img(
    anat_img,
    bg_img=False,
    cmap="gray",
    colorbar=False,
    symmetric_cmap=False,
    title="Click to explore the brain",
)

anat_viewer

### Brain Treasure Hunt

Use the interactive viewer to move near the **upper/parietal cortex:** `(x=-40, y=-25, z=60)`. This is high up, on the surface of the brain. The coordinates are clues, not exact borders: brain areas are larger than one point and differ between people.

Describe the location using ordinary words such as **left/right**, **front/back**, and **top/bottom**. Do not worry about finding one perfect voxel.

## Load an example fMRI scan

To explore real fMRI data, we will download a small example scan. The next cell may take a little longer because the data need to be downloaded the first time.

In [ ]:
# Download one participant's fMRI image and events table.
fmri_dataset = datasets.fetch_localizer_first_level()
fmri_img = image.load_img(fmri_dataset.epi_img)
events = pd.read_table(fmri_dataset.events)

print("fMRI image shape:", fmri_img.shape)
print("Number of dimensions:", len(fmri_img.shape))

## fMRI adds a fourth dimension: time

An anatomical image stores one value at every `(x, y, z)` voxel position. An fMRI image adds a fourth dimension: `t` for **time**.

We can think about the same 4D data in two useful ways:

1. as a sequence of 3D brain volumes, like frames in a movie, or
2. as a time series for every voxel, showing how its signal changes during the scan.

<img src="https://raw.githubusercontent.com/raimund-buehler/visualizing-the-brain/main/src/images/fmri_4d_timeseries.png" alt="A sequence of fMRI volumes and a time series from one voxel" width="700">

The shape `(53, 63, 46, 128)` means:

- 53 voxel positions along the first spatial direction,
- 63 along the second,
- 46 along the third, and
- 128 time points.

The first three numbers describe one 3D volume. The fourth number tells us how many volumes were recorded over time.

## Stop and Think

Imagine you have a Rubik's Cube on the table in front of you. You take a photo of it, turn one side of the cube, and take another photo. You repeat this ten times. At the end, you have 10 photos, or 10 time points.

1. Now you print all photos and line them up. If you look at the figure above, what would the individual photos correspond to in the fMRI image? What would a time series for one voxel be?
2. Which "values" would you look at on the cube over time, meaning across the different photos, if you only looked at one small square?

<details>
<summary>Possible solution</summary>

1. The individual photos correspond to the individual **3D fMRI volumes** over time. At each time point, there is a new image of the brain. A time series for one voxel would be like following the same small square on the cube and watching how its value changes from photo to photo.

2. For the Rubik's Cube, the value could be the **color** of that square: red, blue, green, and so on. In an fMRI image, the value is not a color, but a **signal intensity**. The plot color is created later so that we can see these numbers more easily.

The analogy is not perfect: a Rubik's Cube is visible from the outside, while an fMRI volume contains many voxels inside the brain. But the basic idea is similar: we follow values at specific positions across several time points.

</details>

### Extract one voxel's time series

The figure above shows a line for one voxel. Now we create such a line from real data.

To choose one voxel, we select an `x`, `y`, and `z` array position.

We again choose a voxel near the middle of the image.

In [ ]:
# Choose the middle array position in the fMRI image.
voxel_x = fmri_img.shape[0] // 2
voxel_y = fmri_img.shape[1] // 2
voxel_z = fmri_img.shape[2] // 2

# Read this voxel at every time point.
voxel_signal = np.asarray(
    fmri_img.dataobj[voxel_x, voxel_y, voxel_z, :]
)

print("Voxel index:", (voxel_x, voxel_y, voxel_z))
print("Time-series shape:", voxel_signal.shape)
print("First 10 signal values:", voxel_signal[:10])

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(voxel_signal, color="royalblue")
plt.xlabel("Time point (volume number)")
plt.ylabel("fMRI signal intensity")
plt.title("Signal over time from one voxel")
plt.grid(alpha=0.25)
plt.show()

### Try another voxel

One voxel gives us one time series. But the brain image contains many thousands of voxels. We move a little bit away from the middle voxel and compare the signal.

In the next cell, you can change the `+ 5` values. Keep the changes small, such as `+ 3`, `+ 5`, or `- 5`, so you stay near the original voxel.

In [ ]:
# Try another nearby voxel.
# Change these small shifts and rerun the cell.
new_voxel_x = voxel_x + 5
new_voxel_y = voxel_y
new_voxel_z = voxel_z

new_voxel_signal = np.asarray(
    fmri_img.dataobj[new_voxel_x, new_voxel_y, new_voxel_z, :]
)

plt.figure(figsize=(10, 4))
plt.plot(voxel_signal, color="royalblue", label="middle voxel")
plt.plot(new_voxel_signal, color="darkorange", label="nearby voxel")
plt.xlabel("Time point (volume number)")
plt.ylabel("fMRI signal intensity")
plt.title("Two nearby voxel time series")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

print("Middle voxel:", (voxel_x, voxel_y, voxel_z))
print("Nearby voxel:", (new_voxel_x, new_voxel_y, new_voxel_z))

### Discuss

1. Do the two lines look identical, similar, or very different?
2. Why might nearby voxels have different signals?

<details>
<summary>Possible answers</summary>

Nearby voxels can differ because they may contain different tissue, different amounts of noise, or different mixtures of brain signals. A real fMRI analysis therefore checks many voxels and compares different task conditions.

</details>

### What is the fMRI signal?

At every time point, the scanner stores a signal intensity for every voxel. Part of this number is influenced by the **BOLD signal**. BOLD stands for **blood-oxygen-level dependent**.

When a group of brain cells becomes more active, it needs energy. The body changes the local supply of oxygen-rich blood. fMRI is sensitive to these blood-oxygen changes. It does **not** measure thoughts or electrical brain activity directly.

A voxel's line can also change because of breathing, heartbeat, head movement, scanner noise, and slow changes during the scan. Therefore, one bump in one time series is not enough to claim that a task activated that voxel.

## One single fMRI volume

The time series used one voxel across all 128 time points. We can also do the opposite: choose one time point and look at every voxel in this 3D volume.

An fMRI volume looks blurrier and less detailed than the anatomical image because functional images are collected quickly, over and over again.

In [ ]:
first_volume = image.index_img(fmri_img, 0)

plotting.plot_epi(
    first_volume,
    title="The first 3D volume in the fMRI scan",
)
plotting.show()

### What do the colors mean?

In this image, brighter and darker colors show the **raw signal intensity** recorded in the voxels at one moment.

These colors do **not** show task activation yet. The intensity also depends on tissue type, scanner sensitivity, image position, and other sources of variation.

To say something about a task, we need to connect the changing voxel signal to the experiment and compare **one condition against another condition**.

## The localizer experiment

To compare task conditions, we use data from a so-called "localizer study" by Pinel and colleagues. During a short scan, the participant saw or heard many brief instructions and stimuli.

<img src="https://raw.githubusercontent.com/raimund-buehler/visualizing-the-brain/main/src/images/pinel_localizer_tasks_cropped.png" alt="Visual and audio motor instruction tasks in the Pinel localizer experiment" width="200" style="width:200px; max-width:100%; height:auto;">

One task involved pressing a button with the left or right hand after seeing or hearing an instruction.

If you have never seen an fMRI study before, this task might look a little random. But it is chosen carefully and helps researchers ask a question about an important brain function: How does the brain control our hands, or how does the brain make it possible for us to move at all?

## Connect the scan to the experiment

The scanner only records numbers. It does not know whether the participant was reading, listening, or pressing a button.

The **events table** supplies that missing information. Each row describes one event:

- `trial_type`: what the participant did,
- `onset`: when it began, in seconds after the scan started,
- `duration`: how long it lasted, in seconds.

The analysis uses these timings to line up the experiment with the 128 measured brain volumes.

In [ ]:
# Show only the button-press events we will compare later.
button_press_events = events[
    events["trial_type"].str.contains("hand_button_press")
]

button_press_events[["trial_type", "onset", "duration"]].head(10)

## Build the model

Now we want to connect two things:

1. the fMRI signal from each voxel, and
2. the timing of the task conditions.

To do that, we need to build a model. Our goal is simple: we want to find where activity **for the right hand** was stronger than **for the left hand**. We let Nilearn handle the detailed work for us.

In [ ]:
# Describe and fit one participant's first-level model.
first_level_model = FirstLevelModel(
    t_r=2.4,
    slice_time_ref=0,
    hrf_model="spm",
    drift_model="cosine",
    high_pass=0.01,
    smoothing_fwhm=4,
    minimize_memory=False,
)

first_level_model = first_level_model.fit(fmri_img, events=events)
print("Model fitted successfully.")

## A contrast: right hand compared with left hand

To find out where activity for right-hand button presses was stronger than for left-hand button presses, we need to calculate a **contrast**. This simply means **subtracting** the two conditions.

In short:

```text
right hand - left hand
```

The code below does this for us. Audio and visual instructions are combined.

In [ ]:
# Average across visual and audio instructions for each hand.
right_minus_left = (
    "0.5 * (visual_right_hand_button_press "
    "+ audio_right_hand_button_press) "
    "- 0.5 * (visual_left_hand_button_press "
    "+ audio_left_hand_button_press)"
)

right_minus_left_map = first_level_model.compute_contrast(
    right_minus_left,
    output_type="z_score",
)

print("Contrast calculated.")

### Visualize the contrast

In [ ]:
# Find the strongest right-hand-greater voxel and convert it to map coordinates.
contrast_data = right_minus_left_map.get_fdata()
peak_index = np.unravel_index(np.nanargmax(contrast_data), contrast_data.shape)
peak_coord = right_minus_left_map.affine @ [*peak_index, 1]
peak_coord = peak_coord[:3]

plotting.plot_stat_map(
    right_minus_left_map,
    bg_img=anat_img,
    display_mode="ortho",
    cut_coords=peak_coord,
    threshold=3.0,
    symmetric_cbar=True,
    title="Focused view: strongest cluster for right hand > left hand",
)
plotting.show()

plotting.plot_stat_map(
    right_minus_left_map,
    bg_img=anat_img,
    display_mode="z",
    cut_coords=[40, 50, 60, 70],
    threshold=3.0,
    symmetric_cbar=True,
    title="Right-hand button press minus left-hand button press",
)
plotting.show()

Look carefully at the sides of the brain:

1. Where in the brain do we see the strongest activity? Top or bottom?
2. Where are the warm colors stronger? More on the left or right side of the brain?
3. Where are the cool colors stronger? More on the left or right side of the brain?

### What did we just discover?

The contrast gives us important information about how the brain controls the body.

<img src="https://raw.githubusercontent.com/raimund-buehler/visualizing-the-brain/main/src/images/Anatomytool_Sensory_homunculus_English.jpg" alt="Motor homunculus showing body-part representation along the motor cortex" width="650">

This picture is called a **motor homunculus**. It is a simplified map of the motor cortex. Different parts of this brain area are connected to movements of different body parts. The hand has a large area because hands can make very precise movements.

In our contrast, we compared:

```text
right-hand button press - left-hand button press
```

That means the colors should be read like this:

- **Warm colors** mean: more activity for the **right hand** than for the left hand.
- **Cool colors** mean: less activity for the right hand than for the left hand. In other words, these voxels responded more for the **left hand**.

We just discovered an important principle of brain organization: **contralaterality**. This means that one side of the brain is mainly connected to the opposite side of the body. The left motor cortex is responsible for moving the right hand. The right motor cortex is strongly responsible for moving the left hand.

Such crossings are common in the brain. For example, the left side of visual perception is processed mostly by the right side of the brain, and the right side mostly by the left side.

So the result goes beyond simple button presses if we interpret it correctly: the brain is often organized **contralaterally**.

## Sources and further reading

This notebook adapts teaching material and figures from `TEWA2/02_Understanding_MRI_Data.ipynb` and `TEWA2/06_First_Level_Analysis.ipynb` in this repository.

- [Nilearn: 3D and 4D images](https://nilearn.github.io/stable/auto_examples/00_tutorials/plot_3d_and_4d_niimg.html)
- [Nilearn: first-level models](https://nilearn.github.io/stable/glm/first_level_model.html)
- [Nilearn plotting tools](https://nilearn.github.io/stable/modules/plotting.html)
- Pinel, P., Thirion, B., Meriaux, S. et al. (2007). Fast reproducible identification and large-scale databasing of individual functional cognitive networks. *BMC Neuroscience, 8*, 91. https://doi.org/10.1186/1471-2202-8-91